# Capstone 3.C: Context budget

Your pipeline lives in `/home/elastic/pipeline.py`. Edit it in the **Editor** tab,
and use this notebook to try single questions while you work.

When a change looks right, run the whole dev pool from the Terminal tab:

```
/home/elastic/.venv/bin/python /home/elastic/submit.py
```

It prints the numbers the Check measures. The Check runs your pipeline itself, so
save `pipeline.py` before you select Check.


## The harness and the dev pool


In [ ]:
import importlib, sys
sys.path.insert(0, '/opt/ara/lib')
import ara_capstone as H

pool = H.load_dev_pool()
print(f'{len(pool)} dev questions, budget {H.CONTEXT_BUDGET:,} tokens per question')
for query_class in ('policy', 'case', 'sar', 'unanswerable'):
    example = next(q for q in pool if q['query_class'] == query_class)
    print()
    print(query_class, '|', example['query_id'])
    print('  ', example['query_text'])
    print('   intent:', example['intent'])
    print('   relevant documents:', len(example['relevant_ids']))


## One question through your pipeline

Re-run this cell after every edit to `pipeline.py`. It loads the file from disk each
time, so you never have to restart the kernel.


In [ ]:
import importlib.util

def load_pipeline():
    spec = importlib.util.spec_from_file_location('pipeline', '/home/elastic/pipeline.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

pipeline = load_pipeline()

question = next(q for q in pool if q['query_class'] == 'sar')
row = pipeline.answer_question(question['query_text'], question['query_class'],
                               question['intent'])

print(f"context tokens: {row['context_tokens']:,} of {H.CONTEXT_BUDGET:,}")
print('retrieved      :', len(row['passage_ids']))
print('precision@10  :', H.precision_at_k(row['passage_ids'], question['relevant_ids']))
print('hook fired    :', row['hook'] or 'none')
print()
print(row['answer'][:600])


## Where the tokens go

Retrieval is not the expensive part. Look at what each retrieved unit costs before you
decide how many of them can reach the model.


In [ ]:
index, body = pipeline.route(question['query_text'], question['query_class'],
                             question['intent'])
passages = H.search(index, body, query_class=question['query_class'])
print('index:', index)
total = 0
for p in passages:
    tokens = H.token_count(p['text'])
    total += tokens
    print(f"  {p['passage_id']:<28} {tokens:>6,} tokens  score {p['score']:.3f}")
print(f'  {"total":<28} {total:>6,} tokens  (budget {H.CONTEXT_BUDGET:,})')


## Attribution on one answer

`attribute_by_overlap` is deterministic and calls no model, so the same answer over the
same passages always produces the same map. Read the `UNSUPPORTED` rows: those are the
claims your output validation has to deal with before the answer leaves the pipeline.


In [ ]:
claims = H.split_claims(row['answer'])
for record in H.attribute_by_overlap(claims, passages):
    print(f"  {record['passage_id']:<28} {record['score']:.2f}  {record['claim'][:90]}")


## Then run the pool

In the Terminal tab:

```
/home/elastic/.venv/bin/python /home/elastic/submit.py --limit 3   # fast loop, three per class
/home/elastic/.venv/bin/python /home/elastic/submit.py             # the full forty
```
